**Simulação Computacional - Projecto Mecânico - Aula Prática 3**

<br>

### Exemplo 2.2

<br>

![Imagem:](https://drive.google.com/uc?export=view&id=1126SMiiPdaslD7tnh7IeZaB6Jococ4-Y)


```text
   (Fixo)                                                   (Deslocamento)
                                             
    Nodo 1 --[k]-- Nodo 2 --[k]-- Nodo 3 --[k]-- Nodo 4 --[k]-- Nodo 5
      |              |              |              |              |
   f1x,d1x        f2x,d2x         f3x,d3x       f4x,d4x        f5x,d5x
```

Para esta montagem de molas vamos obter (a) a matriz de rigidez global, (b) o deslocamento dos nodos 2,3 e 4,(c) as forças globais nos nodos, e (d) as forças em cada elemento. Um deslocamento de 20 mm foi aplicado na direção do eixo dos x. As molas têm constantes de rigidez iguais, k = 200 kN/m.

NOTA:
O eixo dos xx local coincide com o eixo dos xx global pelo que não há lugar a transformações entre sistemas de coordenadas.

Aqui introduzimos uma condição de fronteira que é um deslocamento.

In [ ]:
import sympy as sp
from sympy import symbols, Matrix, zeros, solve, Equality
from IPython.display import display

# Define variáveis simbólicas
k_sym = symbols('k')
d1x, d2x, d3x, d4x, d5x = symbols('d1x d2x d3x d4x d5x')
F1x, F2x, F3x, F4x, F5x = symbols('F1x F2x F3x F4x F5x')

# --- (a) Montagem da Matriz de Rigidez Global ---
# Mapeamento de nós (da esquerda para a direita no diagrama):
# Nó 1 do diagrama -> d1x (fixo)
# Nó 2 do diagrama -> d2x (livre)
# Nó 3 do diagrama -> d3x (livre)
# Nó 4 do diagrama -> d4x (livre)
# Nó 5 do diagrama -> d5x (deslocamento aplicado)

# Matriz de rigidez do elemento para uma única mola conectando os nós i e j
def get_element_stiffness(k_val):
    return Matrix([[k_val, -k_val], [-k_val, k_val]])

# Inicializa a matriz de rigidez global (5x5 para 5 nós)
K_global = zeros(5, 5)

# Adiciona contribuições de cada elemento de mola usando a superposição
# Nota que [i,j], onde i > linha, j>coluna
# Mola 1 (k) conecta d1x e d2x
K_global[0, 0] += get_element_stiffness(k_sym)[0,0]
K_global[0, 1] += get_element_stiffness(k_sym)[0,1]
K_global[1, 0] += get_element_stiffness(k_sym)[1,0]
K_global[1, 1] += get_element_stiffness(k_sym)[1,1]

# Mola 2 (k) conecta d2x e d3x
K_global[1, 1] += get_element_stiffness(k_sym)[0,0]
K_global[1, 2] += get_element_stiffness(k_sym)[0,1]
K_global[2, 1] += get_element_stiffness(k_sym)[1,0]
K_global[2, 2] += get_element_stiffness(k_sym)[1,1]

# Mola 3 (k) conecta d3x e d4x
K_global[2, 2] += get_element_stiffness(k_sym)[0,0]
K_global[2, 3] += get_element_stiffness(k_sym)[0,1]
K_global[3, 2] += get_element_stiffness(k_sym)[1,0]
K_global[3, 3] += get_element_stiffness(k_sym)[1,1]

# Mola 4 (k) conecta d4x e d5x
K_global[3, 3] += get_element_stiffness(k_sym)[0,0]
K_global[3, 4] += get_element_stiffness(k_sym)[0,1]
K_global[4, 3] += get_element_stiffness(k_sym)[1,0]
K_global[4, 4] += get_element_stiffness(k_sym)[1,1]

print("\n(a) Matriz de Rigidez Global [K]:")
display(K_global)

#display(K_global.subs({k_sym: 200}))
#-----------------------------------------------------------------------------

# Vamos agora procurar os deslocamentos desconhecidos usando as condições de fronteira.
# d1x = 0 (porque este nó está fixo)
# d5x = 20 mm = 0.02 m (porque é o deslocamento imposto)
# F2x = F3x = F4x = 0 (em equilíbrio as forças resultantes nestes nós são nulas)

# O sistema que temos de resolver é o seguinte:
U_global = Matrix([0, d2x, d3x, d4x, 0.02])
F_global = Matrix([F1x, 0, 0, 0, F5x])

#print("\n(b) Sistema Global:")
#display(K_global, U_global, F_global)

# Tal como fizemos antes, vamos trabalhar com o sistema reduzido, usando as forças conhecidas para calcular os deslocamentos desconhecidos

K_reduced = K_global[1:4, :]
U_unknown = U_global[:, 0] # todos os deslocamentos
F_known = F_global[1:4, 0] # Forças conhecidas nos nós 2-4

print("\nSistema reduzido de equações:")
#display(K_reduced,U_unknown,F_known)
equations = K_reduced * U_unknown - F_known
display(equations)

print("\n(b) Deslocamentos sesconhecidos (d2x, d3x e d4x) em m:")
solution_d = solve(equations, (d2x, d3x, d4x))
display(solution_d)

#-----------------------------------------------------------------------------

# As forças globais dos nodos resolvem-se por substituição dos deslocamentos calculados no sistema global que representa este problema
print("\nSistema global de equações:")
global_equations = K_global * U_global - F_global
solution_d[k_sym] = 200
global_equations = global_equations.subs(solution_d)
display(global_equations)

print("\n(c) Forças globais nos nós (F1x, F5x) em kN:")
solution_f = solve(global_equations, (F1x, F5x))
display(solution_f)

#-----------------------------------------------------------------------------

# A solução para cada elemento fica assim trivial, basta usar a matriz do elemento e os deslocamentos calculados para os seus nós

f1x, f2x, f3x, f4x, f5x = symbols('f1x f2x f3x f4x f5x')

element_1_k = get_element_stiffness(200)
element_1_u = Matrix([0, solution_d[d2x]])
element_1_f = Matrix([f1x, f2x])
element_1_equations = element_1_k * element_1_u - element_1_f
print("\n(d) Exemplo de forças locais no elemento 1:")
display(element_1_equations,element_1_u,element_1_k)
solution_element_1_f = solve(element_1_equations, (f1x, f2x))
display(solution_element_1_f)


(a) Matriz de Rigidez Global [K]:


Matrix([
[ k,  -k,   0,   0,  0],
[-k, 2*k,  -k,   0,  0],
[ 0,  -k, 2*k,  -k,  0],
[ 0,   0,  -k, 2*k, -k],
[ 0,   0,   0,  -k,  k]])


Sistema reduzido de equações:


Matrix([
[          2*d2x*k - d3x*k],
[ -d2x*k + 2*d3x*k - d4x*k],
[-d3x*k + 2*d4x*k - 0.02*k]])


(b) Deslocamentos sesconhecidos (d2x, d3x e d4x) em m:


...


(d) Exemplo de forças locais no elemento 1:


Matrix([
[-f1x - 1.0],
[ 1.0 - f2x]])

Matrix([
[    0],
[0.005]])

Matrix([
[ 200, -200],
[-200,  200]])

{f1x: -1.00000000000000, f2x: 1.00000000000000}


---

## Exercício de Consolidação

Convido-vos a resolverem o problema P2-11 e P2-14 da página 63 do *FEM First Course - Logan 5th Edition*.

P2-11

![Imagem:](https://drive.google.com/uc?export=view&id=1XkSATrO8RKcq_DO2OgezzJFWwUf9G7A0)


P2-14

![Imagem:](https://drive.google.com/uc?export=view&id=1qW87RbFRgYWuchfPNBsL_0wSRahG6MwV)

Determina os deslocamentos nodais, as forças e as reações em cada elemento.



<br>

### Exemplo 2.3

<br>


![Imagem:](https://drive.google.com/uc?export=view&id=1aZw-8-4AgvoBzUcwaBNryI7MlWZOmJtf)



```text
    (Fixo)    (Barra Móvel Axial)    (Fixo)
      |               |                |         
      |             Nodo 2 --[k2]-- Nodo 3
      |               !!               |
    Nodo 1 --[k1]-- Nodo 2          f3x,d3x
      |               !!
      |             Nodo 2 --[k3]-- Nodo 4
      |               |                |
      |               |              (Fixo)
      |               |                |    
   f1x,d1x          f2x,d2x         f4x,d4x           
```

Para esta montagem de molas vamos obter (a) a matriz de rigidez global. O nó 2 está numa barra rígida pelo que $d_{2x} = d_{2x}^{(1)} = d_{2x}^{(2)} = d_{2x}^{(2)}$ e portanto

NOTA: O eixo dos xx local coincide com o eixo dos xx global pelo que não há lugar a transformações entre sistemas de coordenadas.

In [ ]:
import sympy as sp
from sympy import symbols, Matrix, zeros, solve, Equality
from IPython.display import display

# Define variáveis simbólicas
k1_sym, k2_sym, k3_sym  = symbols('k1 k2 k3')
d1x, d2x, d3x, d4x = symbols('d1x d2x d3x d4x')
F1x, F2x, F3x, F4x = symbols('F1x F2x F3x F4x')

# --- (a) Montagem da Matriz de Rigidez Global ---
# Mapeamento de nós (da esquerda para a direita no diagrama):
# Nó 1 do diagrama -> d1x (fixo)
# Nós 2 do diagrama -> d2x (livres)
# Nó 3 do diagrama -> d3x (fixo)
# Nó 4 do diagrama -> d4x (fixo)

# Matriz de rigidez do elemento para uma única mola conectando os nós i e j
def get_element_stiffness(k_val):
    return Matrix([[k_val, -k_val], [-k_val, k_val]])

# Inicializa a matriz de rigidez global (4x4 para 4 nós)
K_global = zeros(4, 4)

# Adiciona contribuições de cada elemento de mola usando a superposição
# Nota que [i,j], onde i > linha, j>coluna
# Mola 1 (k) conecta d1x e d2x
K_global[0, 0] += get_element_stiffness(k1_sym)[0,0]
K_global[0, 1] += get_element_stiffness(k1_sym)[0,1]
K_global[1, 0] += get_element_stiffness(k1_sym)[1,0]
K_global[1, 1] += get_element_stiffness(k1_sym)[1,1]

# Mola 2 (k) conecta d2x e d3x
K_global[1, 1] += get_element_stiffness(k2_sym)[0,0]
K_global[1, 2] += get_element_stiffness(k2_sym)[0,1]
K_global[2, 1] += get_element_stiffness(k2_sym)[1,0]
K_global[2, 2] += get_element_stiffness(k2_sym)[1,1]

# Mola 3 (k) conecta d2x e d4x
K_global[1, 1] += get_element_stiffness(k3_sym)[0,0]
K_global[1, 3] += get_element_stiffness(k3_sym)[0,1]
K_global[3, 1] += get_element_stiffness(k3_sym)[1,0]
K_global[3, 3] += get_element_stiffness(k3_sym)[1,1]

print("\n(a) Matriz de Rigidez Global [K]:")
display(K_global)

#-----------------------------------------------------------------------------

# Sabendo que os deslocamentos são zero para os nós 1-3-4.

equations_global = K_global * Matrix([0, d2x, 0, 0]) - Matrix([F1x, F2x, F3x, F4x])
print("\nSistema global de equações:")
display(equations_global)

solution_global = solve(equations_global, F2x)
print("\nSolução para d2x com base em F2x:")
display(solution_global)


(a) Matriz de Rigidez Global [K]:


Matrix([
[ k1,          -k1,   0,   0],
[-k1, k1 + k2 + k3, -k2, -k3],
[  0,          -k2,  k2,   0],
[  0,          -k3,   0,  k3]])


Sistema global de equações:


Matrix([
[            -F1x - d2x*k1],
[-F2x + d2x*(k1 + k2 + k3)],
[            -F3x - d2x*k2],
[            -F4x - d2x*k3]])


Solução para d2x:


{F2x: d2x*k1 + d2x*k2 + d2x*k3}


---

## Exercício de Consolidação

Convido-vos a resolverem o problema P2-15 da página 63 do *FEM First Course - Logan 5th Edition*.

![Imagem:](https://drive.google.com/uc?export=view&id=1NJAMoJGancwY7Q3dpSQFWE3COP63Czyq)


Determina os deslocamentos nodais, as forças e as reações em cada elemento.



---

### Minimum Potential Energy Approach

Agora que temos uma visão mais definida do método de elementos finitos vamos procurar uma abordagem mais generalista que os métodos de equilíbrio para a obtenção das equações dos elementos e da matriz de rigidez. Esta nova abordagem, variacional, é baseada em minimizar a energia potencial e é aplicável em materiais elásticos sempre que a física subjacente tenha um conceito de energia potencial aplicável. Este prícipio é mais útil e adaptável aos casos de elementos mais complexos e com mais graus de liberdade, como os elementos planares, axissimétricos, tridimencionais, etc.

A energia potencial total de uma determinada estruture $\pi_p$ é expressa em função dos deslocamentos, $\pi_p = \pi_p(d_1, d_2, ..., d_n)$. Quando $\pi_p$ é minimizado com respeito a esses deslocamentos, isso resulta nas equações de equilíbrio. Então, a energia potencial total de uma estrutura é definida como a soma da energia interna de deformação $U$ e a energia potencial das forças externas $Ω$,

$$
\pi_p = U + \Omega
$$

Voltamos ao exemplo do elemento mais simples, a mola linear, onde a força se relaciona com a deformação por $F=kx$, onde $k$ é a constante da mola, e $x$ a deformação provocada na mola.

O diferencial da energia interna de deformação $dU$ provocada por um pequena alteração no comprimento da mola pode ser dada por:

$$
dU = Fdx
$$

Onde $F$ é a força como vimos anterioremente, e logo,

$$
dU = kx dx
$$

E a energia total interna de deformação é:

$$
U = \int_{0}^{x} kx dx = \frac{1}{2} k x² = \frac{1}{2} Fx
$$

O que nos indica que a energia potencial de deformação é a área por baixo da curva de força / deformação.

Por outro aspecto, a energia potencial de uma força externa opõe-se ao trabalho feito por uma força externa e é dado por:

$$
\Omega = -F_{ext}x
$$

E portanto, para uma mola linear, o total de energia potencial é dado por:

$$
\pi_p = \frac{1}{2}kx² - F_{ext}x
$$

O valor estacionário de uma qualquer função, exemplo $G$, (estamos á procura do mínimo) é dado pelos pontos onde a sua derivada é nula $\frac{dG}{dx}=0$, e podem corresponder a mínimos, máximos, ou valores estacionários intermédios (neutros). Para um elemento mola, considerando os dois nós $d_{1x}$ e $d_{2x}$ temos a sua total energia potencial dada por:

$$
\pi_p = \frac{1}{2} k (d_{2x}²-2d_{2x}d_{1x}+d_{1x}²) -f_{1x}d_{1x} - f_{2x}d_{2x}
$$

A minimização de $\pi_p$ em relação a cada deslocamento nodal requer que se tirem derivadas parciais em relação a cada um, tal que,

$$
\frac{\delta \pi_p}{\delta d_{1x}} = \frac{1}{2}k(-2d_{2x}+2d_{1x})-f_{1x}=0
$$

$$
\frac{\delta \pi_p}{\delta d_{2x}} = \frac{1}{2}k(2d_{2x}-2d_{1x})-f_{2x}=0
$$

O que simplificado dá,

$$
k(-d_{2x}+d_{1x}) = f_{1x}
$$

$$
k(-d_{2x}+d_{1x}) = f_{1x}
$$

O que em notação matricial dá a mesma matriz de rigidez que calculamos anteriormente para um elemento mola linear, mas agora através da energia potencial mínima.

$$
\begin{pmatrix}
k & -k \\
-k & k \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{2x} \\
\end{pmatrix}
=
\begin{pmatrix}
f_{1x} \\
f_{2x} \\
\end{pmatrix}
$$